# Mission 3 P0 baseline local re-run (TF-IDF + OvR Logistic Regression)

Local (Windows) adaptation of the Colab draft. Executed with `jupyter nbconvert --execute`; cell outputs are the submission log.
Reference: Macro F1 0.5676 (tol ±0.003), N=29,200, 0 load failures.

In [1]:
# Cell 0: config + provenance
import os, sys, platform, hashlib, subprocess, json, time
from pathlib import Path

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUNBUFFERED"] = "1"

REPO = Path(r"C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim")
LABEL_DIR = REPO / "data" / "raw_merged" / "Training" / "2.라벨링데이터" / "TL_서울_구급"
SCRIPT = REPO / "scripts" / "mission3_baseline 최종 수정본.py"
OUT = REPO / "outputs" / "m3_p0_rerun"
OUT.mkdir(parents=True, exist_ok=True)
EXPECTED_N = 29200
REF_MACRO_F1 = 0.5676
TOL = 0.003

def git(*args):
    return subprocess.run(["git", "-C", str(REPO), *args], capture_output=True,
                          text=True, encoding="utf-8", errors="replace").stdout.strip()

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

HEAD = git("rev-parse", "HEAD")
print("git HEAD          :", HEAD)
print("git log -1        :", git("log", "-1", "--oneline"))
st = git("status", "--porcelain", "--", "scripts/")
print("git status scripts:", st if st else "(clean)")
SCRIPT_SHA = sha256(SCRIPT)
print("script            :", SCRIPT)
print("script sha256     :", SCRIPT_SHA)
print("python            :", sys.version.replace("\n", " "))
print("executable        :", sys.executable)
print("platform          :", platform.platform(), "|", platform.processor())
print("LABEL_DIR         :", LABEL_DIR)
n_json = sum(1 for _ in LABEL_DIR.rglob("*.json"))
print("JSON count        :", n_json)
assert n_json == EXPECTED_N, f"expected {EXPECTED_N} JSON, got {n_json}"
print("OUT               :", OUT)


git HEAD          : a1ad0734ac7bca34210f42adf62637b5cfcae668
git log -1        : a1ad073 Merge pull request #45 from eigensinn-dcc2026/chore/gitignore-tmp-attachments
git status scripts: (clean)
script            : C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\scripts\mission3_baseline 최종 수정본.py
script sha256     : fee2bef3d52568fd8ecfb31773f0b8987a3dc4a3517508d858fdac9fc7209103
python            : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
executable        : C:\Users\sooyah\AppData\Local\Programs\Python\Python311\python.exe
platform          : Windows-10-10.0.26200-SP0 | Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
LABEL_DIR         : C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\data\raw_merged\Training\2.라벨링데이터\TL_서울_구급


JSON count        : 29200
OUT               : C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\outputs\m3_p0_rerun


In [2]:
# Cell 1: dependency check (local: install iterative-stratification only if missing)
try:
    import iterstrat
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "iterative-stratification"], check=True)
    import iterstrat
import sklearn, numpy, pandas, scipy, joblib
print("iterstrat", getattr(iterstrat, "__version__", "(no __version__)"), "|", iterstrat.__file__)
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| joblib", joblib.__version__)


iterstrat 0.1.9 | C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\iterstrat\__init__.py
sklearn 1.8.0 | numpy 2.4.4 | pandas 3.0.2 | scipy 1.17.1 | joblib 1.5.3


In [3]:
# Cell 2: streaming run() helper
LOG = OUT / "run_log.txt"

def run(args):
    cmd = [sys.executable, "-u", *map(str, args)]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    print("$", " ".join(f'"{c}"' if " " in c else c for c in cmd), flush=True)
    t0 = time.time()
    with open(LOG, "a", encoding="utf-8") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                env=env, text=True, encoding="utf-8", errors="replace",
                                bufsize=1, cwd=str(REPO))
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        rc = proc.wait()
        el = time.time() - t0
        log.write(f"[returncode={rc} elapsed={el:.1f}s]\n")
    print(f"[returncode={rc} elapsed={el:.1f}s]")
    assert rc == 0, f"command failed with returncode {rc}"
    return el


In [4]:
# Cell 3: train + evaluate P0 baseline
ELAPSED = run([SCRIPT, "--label_dir", LABEL_DIR, "--output_dir", OUT,
               "--test_size", "0.2", "--seed", "42", "--threshold", "0.5"])
print(f"training wall time: {ELAPSED:.1f}s ({ELAPSED/60:.1f} min)")


$ C:\Users\sooyah\AppData\Local\Programs\Python\Python311\python.exe -u "C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\scripts\mission3_baseline 최종 수정본.py" --label_dir C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\data\raw_merged\Training\2.라벨링데이터\TL_서울_구급 --output_dir C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\outputs\m3_p0_rerun --test_size 0.2 --seed 42 --threshold 0.5


Train samples: 23,408


Validation samples: 5,792


Precision (macro): 0.5323


Recall (macro): 0.6147


...

   호흡곤란      785   0.575789 0.696815 0.630548


전체 지표


Precision(macro): 0.5323 | Recall(macro): 0.6147


Precision(micro): 0.5363 | Recall(micro): 0.6265


[returncode=0 elapsed=933.4s]
training wall time: 933.4s (15.6 min)


In [5]:
# Cell 4: verdict
import pandas as pd
metrics = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
per_label = pd.read_csv(OUT / "per_label_metrics.csv", encoding="utf-8-sig")
print(per_label.to_string(index=False))

REF_PER_LABEL = {"열상": 0.804, "복통": 0.736, "호흡곤란": 0.631, "어지러움": 0.607, "고열": 0.601,
                 "전신쇠약": 0.523, "구토": 0.486, "두통": 0.409, "오심": 0.312}
f1_col = next((c for c in per_label.columns if "f1" in c.lower()), None)
lab_col = per_label.columns[0]
if f1_col is not None:
    cmp = per_label[[lab_col, f1_col]].copy()
    cmp["ref_f1"] = cmp[lab_col].map(REF_PER_LABEL)
    cmp["diff"] = (cmp[f1_col] - cmp["ref_f1"]).round(4)
    print("\nper-label F1 vs reference:")
    print(cmp.to_string(index=False))

CKPT = OUT / "mission3_tfidf_logreg.joblib"
CKPT_SHA = sha256(CKPT)
print("\nckpt:", CKPT)
print("ckpt sha256:", CKPT_SHA)

macro = float(metrics["macro_f1"])
diff = macro - REF_MACRO_F1
passed = bool(abs(diff) <= TOL and metrics["total_sample_count"] == EXPECTED_N
              and metrics["load_failure_count"] == 0)
verdict = {"macro_f1": macro, "ref": REF_MACRO_F1, "diff": diff, "tol": TOL, "pass": passed,
           "ckpt": str(CKPT), "ckpt_sha256": CKPT_SHA, "repo_commit": HEAD,
           "script_sha256": SCRIPT_SHA, "micro_f1": metrics.get("micro_f1"),
           "train_sample_count": metrics.get("train_sample_count"),
           "validation_sample_count": metrics.get("validation_sample_count"),
           "total_sample_count": metrics.get("total_sample_count"),
           "load_failure_count": metrics.get("load_failure_count"),
           "elapsed_sec": round(ELAPSED, 1)}
(OUT / "verdict.json").write_text(json.dumps(verdict, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(verdict, ensure_ascii=False, indent=2))
print(f"\nVERDICT: {'PASS' if passed else 'FAIL'}  macro_f1={macro:.5f} ref={REF_MACRO_F1} diff={diff:+.5f} tol={TOL}")


{
  "train_sample_count": 23408,
  "validation_sample_count": 5792,
  "precision_macro": 0.5323176818092431,
  "recall_macro": 0.6146729338924944,
  "precision_micro": 0.5363375076986245,
  "recall_micro": 0.6264988009592326,
  "macro_f1": 0.567552067774815,
  "micro_f1": 0.5779227961508683,
  "threshold": 0.5,
  "test_size": 0.2,
  "seed": 42,
  "total_sample_count": 29200,
  "source_audio_zip": null,
  "load_failure_count": 0
}


symptom  support  precision   recall       f1
     고열     1037   0.567869 0.637416 0.600636
     구토      893   0.454280 0.522956 0.486205
     두통      581   0.397083 0.421687 0.409015
     복통     1354   0.783333 0.694239 0.736100
   어지러움     1267   0.577334 0.639305 0.606742
     열상      693   0.732779 0.890332 0.803909
     오심      668   0.258604 0.393713 0.312166
   전신쇠약     1062   0.443787 0.635593 0.522648
   호흡곤란      785   0.575789 0.696815 0.630548



per-label F1 vs reference:
symptom       f1  ref_f1    diff
     고열 0.600636   0.601 -0.0004
     구토 0.486205   0.486  0.0002
     두통 0.409015   0.409  0.0000
     복통 0.736100   0.736  0.0001
   어지러움 0.606742   0.607 -0.0003
     열상 0.803909   0.804 -0.0001
     오심 0.312166   0.312  0.0002
   전신쇠약 0.522648   0.523 -0.0004
   호흡곤란 0.630548   0.631 -0.0005

ckpt: C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\outputs\m3_p0_rerun\mission3_tfidf_logreg.joblib
ckpt sha256: d9b9df04f90784b499b5b2cc2eb55d630d899d44bbe35e72fdeaceaa0c30beb9


{
  "macro_f1": 0.567552067774815,
  "ref": 0.5676,
  "diff": -4.793222518495455e-05,
  "tol": 0.003,
  "pass": true,
  "ckpt": "C:\\Users\\sooyah\\Desktop\\아주대\\dcc\\01_prelim\\dcc2026-prelim\\outputs\\m3_p0_rerun\\mission3_tfidf_logreg.joblib",
  "ckpt_sha256": "d9b9df04f90784b499b5b2cc2eb55d630d899d44bbe35e72fdeaceaa0c30beb9",
  "repo_commit": "a1ad0734ac7bca34210f42adf62637b5cfcae668",
  "script_sha256": "fee2bef3d52568fd8ecfb31773f0b8987a3dc4a3517508d858fdac9fc7209103",
  "micro_f1": 0.5779227961508683,
  "train_sample_count": 23408,
  "validation_sample_count": 5792,
  "total_sample_count": 29200,
  "load_failure_count": 0,
  "elapsed_sec": 933.4
}

VERDICT: PASS  macro_f1=0.56755 ref=0.5676 diff=-0.00005 tol=0.003


In [6]:
# Cell 5: pip freeze
import re
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True,
                        text=True, encoding="utf-8", errors="replace").stdout
(OUT / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
print("wrote", OUT / "pip_freeze.txt", f"({len(freeze.splitlines())} packages)")
for line in freeze.splitlines():
    if re.search(r"scikit-learn|numpy|pandas|scipy|joblib|iterative", line, re.I):
        print(line)


wrote C:\Users\sooyah\Desktop\아주대\dcc\01_prelim\dcc2026-prelim\outputs\m3_p0_rerun\pip_freeze.txt (203 packages)
iterative-stratification==0.1.9
joblib==1.5.3
numpy==2.4.4
pandas==3.0.2
pandas-datareader==0.10.0
scikit-learn==1.8.0
scipy==1.17.1
